# B2-rand — test seed 44, resumable, 9-hour session budget

Upload `grace_b2_rand_test_seed44_bundle.zip` as a Kaggle Dataset and attach it plus the same pinned GGUF model used for validation. Select GPU T4 x2 and Internet ON.
This notebook runs only B2-rand test for prompt seed 44. It uses the per-dataset thresholds selected on seed44 validation and locked before test inference; test labels are never used to choose thresholds.
Kaggle may recursively unpack the uploaded bundle. The discovery cell accepts the intact outer ZIP or already extracted nested ZIPs, verifies each file's SHA-256, and normalizes extracted members before the runner reads them.
The 9-hour wall-clock budget includes package/CUDA setup and model loading. One record is fsynced after each completed ID; a 5-minute reserve stops new work and exports a checkpoint.
To resume after a disconnect, download `grace_b2_rand_seed44_test_checkpoint.zip` from Kaggle Output, attach it as an Input Dataset, and rerun from the top. The runner verifies prompt, model, threshold, and runtime identities, then skips verified completed IDs.

Validation provenance: the Devign validation threshold includes 2,426 records carried forward from I-15 under the reviewed legacy-resume migration; those historical rows retain their original provenance. The remaining 306 Devign rows and all 2,273 REVEAL rows were scored under the runtime lock. See the attached threshold lock and worklog.


> Future Kaggle runs use the shared package lock (SHA-256 `bf1f859dc786e206a3c11d655b04b528dc68f59a15a82d4db894ac5944b0ac2e`), including SciPy 1.16.3. Existing outputs in this notebook document earlier runs and are not retroactively changed.


In [ ]:
from pathlib import Path
import sys, os, json, subprocess, traceback, shutil, platform, importlib.metadata, zipfile, hashlib, time, re
from datetime import datetime, timezone

SESSION_START_EPOCH = time.time()
SESSION_DEADLINE_EPOCH = SESSION_START_EPOCH + 9 * 3600
DIAG = Path('/kaggle/working/grace_diagnostics')
DIAG.mkdir(parents=True, exist_ok=True)
EVENTS = DIAG / 'events.log'
_gpu_probe = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
                            check=True, capture_output=True, text=True)
GPU_NAMES = [x.strip() for x in _gpu_probe.stdout.splitlines() if x.strip()]
if len(GPU_NAMES) != 2 or any('T4' not in name.upper() for name in GPU_NAMES):
    raise RuntimeError(f'Requires exactly 2 Kaggle T4 GPUs; detected {GPU_NAMES}')
print(f'Using 2 GPUs: {GPU_NAMES}', flush=True)

def note(message):
    text = str(message)
    print(text, flush=True)
    with EVENTS.open('a', encoding='utf-8') as f:
        f.write(f'[{datetime.now(timezone.utc).isoformat()}] {text}\n')

def diagnostics_zip():
    archive = shutil.make_archive('/kaggle/working/grace_diagnostics', 'zip', root_dir=DIAG)
    print('Tải log chẩn đoán từ Kaggle Output:', archive, flush=True)
    return archive

def run_logged(cmd, name, env=None):
    log = DIAG / f'{name}.log'
    note(f'Bắt đầu {name}: {cmd!r} | log đầy đủ: {log}')
    is_inference = name.startswith('test_seed_')
    with log.open('w', encoding='utf-8') as f:
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                text=True, encoding='utf-8', errors='replace', bufsize=1, env=env)
        import threading
        def terminate_at_deadline():
            if proc.poll() is None: proc.terminate()
        watchdog = threading.Timer(max(1, SESSION_DEADLINE_EPOCH-time.time()-120), terminate_at_deadline)
        watchdog.daemon = True
        watchdog.start()
        for line in proc.stdout:
            f.write(line); f.flush()
            if re.search(r'\b(?:Found|Skip(?:ping)?) link\b', line, flags=re.IGNORECASE):
                continue
            if is_inference:
                match = re.search(r'^(\S+).*\((\d+)/(\d+)\)\s*$', line.strip())
                if match:
                    run_key, done, total = match.group(1), int(match.group(2)), int(match.group(3))
                    if done % 100 == 0 or done == total:
                        print(f'Inference {run_key}: {done}/{total}', flush=True)
                elif line.strip():
                    print(line, end='', flush=True)
            else:
                print(line, end='', flush=True)
        code = proc.wait()
        watchdog.cancel()
    if is_inference and '--status' in cmd:
        status_path = Path(cmd[cmd.index('--status') + 1])
        if status_path.is_file():
            status = json.loads(status_path.read_text(encoding='utf-8'))
            print(f"Inference summary: {status.get('completed_records_total', 0)} records; state={status.get('state')}", flush=True)
            for run_key, progress in status.get('runs', {}).items():
                print(f"  {run_key}: {progress.get('completed', 0)}/{progress.get('count', '?')} [{progress.get('state', 'pending')}]")
    note(f'{name} kết thúc với mã {code}')
    if code:
        raise subprocess.CalledProcessError(code, cmd)

def versions(label='Installed package versions'):
    result = {}
    for name in ('llama-cpp-python', 'transformers', 'huggingface_hub', 'scikit-learn', 'numpy', 'scipy'):
        try: result[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError: result[name] = 'not installed'
    note(label + ': ' + json.dumps(result))
    return result

def capture_failure(shell, etype, evalue, tb, tb_offset=None):
    trace = ''.join(traceback.format_exception(etype, evalue, tb))
    try:
        with (DIAG / 'exceptions.log').open('a', encoding='utf-8') as f:
            f.write(f'\n[{datetime.now(timezone.utc).isoformat()}]\n{trace}\n')
        note(f'Lỗi: {etype.__name__}: {evalue}')
        versions()
        diagnostics_zip()
    except Exception as diagnostic_error:
        print('Không thể ghi đầy đủ chẩn đoán:', diagnostic_error, flush=True)
    shell.showtraceback((etype, evalue, tb), tb_offset=tb_offset)
    return []

get_ipython().set_custom_exc((Exception,), capture_failure)
note('UTC: ' + datetime.now(timezone.utc).isoformat())
note('Python: ' + sys.version)
note('Executable: ' + sys.executable)
note('Platform: ' + platform.platform())
for loc in ('/kaggle/working', '/kaggle/input'):
    try:
        usage = shutil.disk_usage(loc)
        note(f'Disk {loc}: total={usage.total} free={usage.free} bytes')
    except OSError as exc: note(f'Disk {loc}: {exc}')
versions('Kaggle base image package versions before pinned installation')
for name, cmd in [('gpu', ['nvidia-smi']), ('cuda_compiler', ['nvcc', '--version']), ('cmake', ['cmake', '--version']), ('pip', [sys.executable, '-m', 'pip', '--version'])]:
    try: run_logged(cmd, name)
    except (OSError, subprocess.CalledProcessError) as exc: note(f'{name}: {type(exc).__name__}: {exc}')
diagnostics_zip()

# Approved shared runtime contract, embedded byte-for-byte for single-file Kaggle upload.
RUNTIME_LOCK_BYTES = b'{\n  "backend_source": {\n    "allow_other_backend": false,\n    "name": "llama-cpp-python",\n    "record_runner_source_sha256": true,\n    "sdist_filename": "llama_cpp_python-0.3.16.tar.gz",\n    "sdist_sha256": "34ed0f9bd9431af045bb63d9324ae620ad0536653740e9bb163a2e1fcb973be6",\n    "verify_sdist_before_build": true,\n    "version": "0.3.16"\n  },\n  "build": {\n    "CMAKE_BUILD_PARALLEL_LEVEL": "2",\n    "CMAKE_CUDA_ARCHITECTURES": "75",\n    "CMAKE_LIBRARY_PATH": "/usr/local/nvidia/lib64",\n    "FORCE_CMAKE": "1",\n    "GGML_CUDA": true,\n    "binary_environment_fully_locked": false,\n    "compiled_wheel_sha256": null,\n    "exact_cmake": null,\n    "exact_cuda_toolkit": null,\n    "exact_host_compiler": null,\n    "exact_python_patch": null,\n    "policy": "Kaggle platform build is approved. Record exact Python patch, CUDA/nvcc/driver, compiler, CMake, flags, dependencies and compiled backend hash. A shared container or bit-identical binary is not required or claimed.",\n    "python_family": "3.13"\n  },\n  "constructor": {\n    "embedding": false,\n    "flash_attn": false,\n    "logits_all": true,\n    "main_gpu": 0,\n    "n_batch": 256,\n    "n_ctx": 8192,\n    "n_gpu_layers": -1,\n    "n_ubatch": 256,\n    "offload_kqv": true,\n    "seed": 4294967295,\n    "split_mode": "LLAMA_SPLIT_MODE_LAYER",\n    "tensor_split": null,\n    "use_mlock": false,\n    "use_mmap": true\n  },\n  "contract_id": "grace_qwen_q4_runtime_v1",\n  "created_on": "2026-11-07",\n  "decoding": {\n    "generation_api": null,\n    "mode": "direct_logits_and_greedy_argmax",\n    "note": "Sampling parameters are not used by eval; the default-seed sentinel does not promise fixed RNG output for future sampling paths.",\n    "sampling": false,\n    "temperature": null,\n    "top_k": null,\n    "top_p": null\n  },\n  "gpu": {\n    "allow_auxiliary_host_buffers": true,\n    "allow_silent_cpu_layer_fallback": false,\n    "device_count": 2,\n    "device_model": "NVIDIA Tesla T4",\n    "distribution": "layer_by_free_vram_at_load",\n    "fixed_ratio_guaranteed": false,\n    "other_hardware_requires_explicit_profile_review": true,\n    "profile": "kaggle_t4_x2",\n    "random_distribution": false,\n    "record_device_order_and_uuid": true,\n    "record_free_and_total_vram": true,\n    "record_layer_and_buffer_offload_report": true,\n    "required_offloaded_layers": 49\n  },\n  "integration": {\n    "existing_notebooks_auto_enforce_json": false,\n    "historical_outputs_retroactively_certified": false,\n    "historical_runs_are_not_retroactively_certified": true,\n    "package_contract_applies_to_all_registered_kaggle_arms": true,\n    "required_before_claiming_compliance": "load, validate and apply this JSON; abort on incompatible values"\n  },\n  "integrity": {\n    "algorithm": "sha256",\n    "canonicalization": "UTF-8 json.dumps(sort_keys=True,separators=(comma,colon),ensure_ascii=False); exclude integrity",\n    "sha256": "d79396de5d37ca6c3123ac05ea122d665f5f9b252aa0b32ac2b5c464b68e338f"\n  },\n  "model": {\n    "adapters_allowed": false,\n    "filename": "Qwen3-Coder-30B-A3B-Instruct-Q4_K_M.gguf",\n    "quantization": "Q4_K_M",\n    "repo": "ggml-org/Qwen3-Coder-30B-A3B-Instruct-GGUF",\n    "sha256": "f6050b4da22869aa3fb9d94bb3314e9fc5a027d4741c1f97761c387d6b2b2335",\n    "verify_sha256_before_load": true\n  },\n  "notebook_audit": [\n    {\n      "configured_inference_seed": 4294967295,\n      "file_sha256_at_audit": "d83c14d4f069e1987c024b27ac801840f7fa080d6fd3f75a17f2e004fdce9ba3",\n      "missing_explicit_package_pins": [\n        "numpy",\n        "huggingface-hub",\n        "tokenizers"\n      ],\n      "path": "01_Author_Baseline_Unified/notebooks/02_kaggle_qwen_q4_b2.ipynb",\n      "shared_json_integrated": false,\n      "tensor_split": null\n    },\n    {\n      "configured_inference_seed": 42,\n      "file_sha256_at_audit": "fdc0ee82e6df6903463023471e2babef6fd6a136bfc4ce14e7dd8806376ee7d6",\n      "missing_explicit_package_pins": [\n        "numpy",\n        "huggingface-hub",\n        "tokenizers"\n      ],\n      "path": "02_Arm_B0_Source_Only/notebooks/02_kaggle_qwen_q4_b0.ipynb",\n      "shared_json_integrated": false,\n      "tensor_split": null\n    },\n    {\n      "configured_inference_seed": 42,\n      "file_sha256_at_audit": "c01dac2696b9391414d0f9c5ee0f7eb1f26e1ef0f69de13b953da83f75f18746",\n      "missing_explicit_package_pins": [\n        "numpy",\n        "huggingface-hub",\n        "tokenizers"\n      ],\n      "path": "03_Arm_B1_Graph_Only/notebooks/02_kaggle_qwen_q4_b1.ipynb",\n      "shared_json_integrated": false,\n      "tensor_split": null\n    },\n    {\n      "configured_inference_seed": 42,\n      "file_sha256_at_audit": "6e501ab2f57b964e9c43a807297df4407620bcb9a02f0a3744d742341dce22fb",\n      "missing_explicit_package_pins": [\n        "numpy",\n        "huggingface-hub",\n        "tokenizers"\n      ],\n      "path": "03_Arm_B1_Graph_Only/notebooks/02_kaggle_qwen_q4_b1_executed_I08.ipynb",\n      "shared_json_integrated": false,\n      "tensor_split": null\n    },\n    {\n      "configured_inference_seed": 42,\n      "file_sha256_at_audit": "c6e7ce9c1351ab88375cd6e4d9b7d34cd5bb9f7862108fef5d12d81196cf37fb",\n      "missing_explicit_package_pins": [\n        "numpy",\n        "huggingface-hub",\n        "tokenizers"\n      ],\n      "path": "04_Arm_B2_Flip/notebooks/02_kaggle_qwen_q4_b2_flip.ipynb",\n      "shared_json_integrated": false,\n      "tensor_split": null\n    },\n    {\n      "configured_inference_seed": 4294967295,\n      "file_sha256_at_audit": "b819048b5f126882dba6ff59581354937101195d8807974bc3a1f7cc65910c86",\n      "missing_explicit_package_pins": [],\n      "path": "06_Arm_B2_Rand/notebooks/02_kaggle_qwen_q4_b2_rand.ipynb",\n      "shared_json_integrated": false,\n      "tensor_split": null\n    },\n    {\n      "configured_inference_seed": 4294967295,\n      "file_sha256_at_audit": "a35b30da7d32ca935f0d5bba894ccb4fa02ba52e1212c70d6afded2ba08e3c7e",\n      "missing_explicit_package_pins": [],\n      "path": "06_Arm_B2_Rand/test_delivery_20261107/02_kaggle_qwen_q4_b2_rand_seed42_test.ipynb",\n      "shared_json_integrated": false,\n      "tensor_split": null\n    },\n    {\n      "configured_inference_seed": 4294967295,\n      "file_sha256_at_audit": "42ebe992c4e716ced593efdd089a720d3d0019663b69f5faf303c201975264d1",\n      "missing_explicit_package_pins": [],\n      "path": "06_Arm_B2_Rand/test_delivery_20261107/02_kaggle_qwen_q4_b2_rand_seed43_test.ipynb",\n      "shared_json_integrated": false,\n      "tensor_split": null\n    },\n    {\n      "configured_inference_seed": 4294967295,\n      "file_sha256_at_audit": "3cb1f4d4fc2d4e9ca839d22db2b1660f671aebbd046bdd2a0e2e94f4a8541dc7",\n      "missing_explicit_package_pins": [],\n      "path": "11_Arm_B1H2_Evidence/kaggle_b1h2_test_20261106/02_kaggle_qwen_q4_b1_h2_test_20261106.ipynb",\n      "shared_json_integrated": false,\n      "tensor_split": [\n        1.0,\n        1.0\n      ]\n    },\n    {\n      "configured_inference_seed": 4294967295,\n      "file_sha256_at_audit": "57cdf5249395aec0f0c5bc47421a9d6fb3789fd5809de7d42bc80f632d91c88f",\n      "missing_explicit_package_pins": [\n        "numpy",\n        "huggingface-hub",\n        "tokenizers"\n      ],\n      "path": "11_Arm_B1H2_Evidence/notebooks/02_kaggle_qwen_q4_b1_h2_validation_20261106.ipynb",\n      "shared_json_integrated": false,\n      "tensor_split": [\n        1.0,\n        1.0\n      ]\n    },\n    {\n      "configured_inference_seed": 4294967295,\n      "file_sha256_at_audit": "dcfff73568daddd1e97934c0f2175d4b36b36339032049fffb16160a541b34bf",\n      "missing_explicit_package_pins": [\n        "numpy",\n        "huggingface-hub",\n        "tokenizers"\n      ],\n      "path": "Runs-management/kaggle_validation_6arms_20261104/02_kaggle_qwen_q4_b1_full_validation.ipynb",\n      "shared_json_integrated": false,\n      "tensor_split": null\n    },\n    {\n      "configured_inference_seed": 4294967295,\n      "file_sha256_at_audit": "d9ed03125e8f309e28b5f06d14be3038e3fe77277fe9ae0ac33857d777aa9759",\n      "missing_explicit_package_pins": [\n        "numpy",\n        "huggingface-hub",\n        "tokenizers"\n      ],\n      "path": "Runs-management/kaggle_validation_6arms_20261104/02_kaggle_qwen_q4_b1_shuf_validation.ipynb",\n      "shared_json_integrated": false,\n      "tensor_split": null\n    },\n    {\n      "configured_inference_seed": 4294967295,\n      "file_sha256_at_audit": "6a54f85e4537e66ab1a1c57615232e3b8a99850a70308c3d1bcadb0fcdb3d8db",\n      "missing_explicit_package_pins": [\n        "numpy",\n        "huggingface-hub",\n        "tokenizers"\n      ],\n      "path": "Runs-management/kaggle_validation_6arms_20261104/02_kaggle_qwen_q4_b2_labelonly_validation.ipynb",\n      "shared_json_integrated": false,\n      "tensor_split": null\n    },\n    {\n      "configured_inference_seed": 4294967295,\n      "file_sha256_at_audit": "099325d9a24e2df72e43a33f77250ea1590302409d7d5cb1b5a7a035f605364a",\n      "missing_explicit_package_pins": [\n        "numpy",\n        "huggingface-hub",\n        "tokenizers"\n      ],\n      "path": "Runs-management/kaggle_validation_6arms_20261104/02_kaggle_qwen_q4_b2_nolabel_validation.ipynb",\n      "shared_json_integrated": false,\n      "tensor_split": null\n    },\n    {\n      "configured_inference_seed": 4294967295,\n      "file_sha256_at_audit": "ccef18aeb640a0ded8d5050e54e58a6e504cd0986460288425fba7a5c7dfed28",\n      "missing_explicit_package_pins": [],\n      "path": "Runs-management/kaggle_validation_6arms_20261104/02_kaggle_qwen_q4_b2_rand.ipynb",\n      "shared_json_integrated": false,\n      "tensor_split": null\n    },\n    {\n      "configured_inference_seed": 4294967295,\n      "file_sha256_at_audit": "2e6bfb53c9fc4525a014a9d79ad862b5c3fbc40fd3bcf8e24729ab91257e7a8e",\n      "missing_explicit_package_pins": [\n        "numpy",\n        "huggingface-hub",\n        "tokenizers"\n      ],\n      "path": "Runs-management/kaggle_validation_6arms_20261104/02_kaggle_qwen_q4_b2_rep_validation.ipynb",\n      "shared_json_integrated": false,\n      "tensor_split": null\n    }\n  ],\n  "open_questions": [\n    {\n      "arms": [\n        "B1-full",\n        "B1-shuf",\n        "B2-rep",\n        "B2-nolabel",\n        "B2-labelonly"\n      ],\n      "id": "missing_run_logs",\n      "status": "awaiting executed notebooks/install logs"\n    },\n    {\n      "arms": [\n        "B3",\n        "B3-nodiff",\n        "B3-swap",\n        "B3-flip"\n      ],\n      "id": "b3_runners",\n      "status": "no dedicated current Kaggle Qwen notebook found in GRACE-summary"\n    }\n  ],\n  "package_contract_scope": "All registered Kaggle Qwen Q4 inference arms (B0, B1, B1-full, B1-H2, B1-shuf, B2, B2-rand, B2-nolabel, B2-labelonly, B2-flip, B2-rep, B3, B3-nodiff, B3-swap, B3-flip). Apply to future runs; historical outputs retain their recorded environment.",\n  "packages": {\n    "huggingface-hub": "0.36.2",\n    "llama-cpp-python": "0.3.16",\n    "numpy": "2.1.3",\n    "scikit-learn": "1.5.2",\n    "scipy": "1.16.3",\n    "tokenizers": "0.22.2",\n    "transformers": "4.57.1"\n  },\n  "prompt_contract": {\n    "different_arms_use_their_own_frozen_prompts": true,\n    "identity": [\n      "arm",\n      "prompt seed",\n      "dataset",\n      "split",\n      "ID",\n      "bundle SHA256",\n      "prompt SHA256"\n    ],\n    "prompt_selection_seeds": [\n      42,\n      43,\n      44\n    ],\n    "selection_seed_distinct_from_inference_seed": true,\n    "silent_reformatting_or_truncation": false,\n    "verify_frozen_input_and_each_prompt_hash": true\n  },\n  "provenance_required": [\n    "contract SHA256",\n    "all package versions including transitive dependencies",\n    "exact Python/OS/libc",\n    "CUDA toolkit/nvcc/driver",\n    "GPU device order/UUID/VRAM",\n    "host compiler/CMake/build flags",\n    "source archive SHA256",\n    "compiled backend/wheel SHA256",\n    "runner source SHA256",\n    "effective CPU threads",\n    "effective batch/microbatch",\n    "offload log",\n    "model/tokenizer/prompt/threshold identities"\n  ],\n  "references": {\n    "auto_split": "https://raw.githubusercontent.com/ggml-org/llama.cpp/4227c9be4268ac844921b90f31595f81236bd317/src/llama-model.cpp",\n    "constructor_defaults": "https://raw.githubusercontent.com/abetlen/llama-cpp-python/v0.3.16/llama_cpp/llama.py",\n    "package_hash": "https://pypi.org/pypi/llama-cpp-python/0.3.16/json"\n  },\n  "resolved_decisions": [\n    {\n      "decision": "Kaggle platform build with mandatory full provenance",\n      "id": "build_artifact",\n      "question": "Pin a shared container/build artifact or allow platform builds with provenance? All notebook builders leave exact CUDA/compiler/CMake/Python patch unpinned.",\n      "status": "resolved"\n    },\n    {\n      "decision": "preserve default CPU-count formulas and record effective counts",\n      "id": "cpu_threads",\n      "interim_policy": "preserve existing formulas and record effective values",\n      "question": "Keep CPU-count default formulas or lock absolute thread counts? All 16 constructors omit n_threads and n_threads_batch.",\n      "status": "resolved"\n    }\n  ],\n  "resume": {\n    "changed_contract": "do not silently append to historical results; use separate run identity or reviewed migration",\n    "require_match": [\n      "arm",\n      "prompt seed",\n      "dataset",\n      "split",\n      "ID",\n      "label",\n      "prompt SHA256",\n      "model SHA256",\n      "tokenizer revision",\n      "backend source",\n      "scoring",\n      "constructor",\n      "threshold lock when applicable"\n    ],\n    "rewrite_historical_outputs": false\n  },\n  "schema_version": 1,\n  "scope": "New GRACE diagnostic Qwen Q4 inference on Kaggle only; excludes Colab/Vast and preparation/CodeT5/legacy API prototypes",\n  "scoring": {\n    "evaluate": "llm.eval(gguf_ids)",\n    "logits": "llm.scores[llm.n_tokens - 1]",\n    "logprob": "stable full-vocabulary log_softmax of last-position logits",\n    "name": "unconstrained_first_token_logits_v1",\n    "normalization_dtype": "float64",\n    "pred_argmax": "int(s >= 0.5)",\n    "raw_token": "argmax of full-vocabulary logits",\n    "reject_non_finite": true,\n    "reset_before_each_prompt": true,\n    "retain_valid_score_if_raw_not_binary": true,\n    "s": "exp(logprob_1 - logaddexp(logprob_0, logprob_1))"\n  },\n  "status": "shared_package_contract_locked_notebook_adoption_in_progress",\n  "threads": {\n    "approval_status": "user_approved",\n    "fixed_thread_count": null,\n    "n_threads": "max(multiprocessing.cpu_count() // 2, 1)",\n    "n_threads_batch": "multiprocessing.cpu_count()",\n    "policy": "preserve_backend_default_formulas",\n    "record_effective_counts": true\n  },\n  "thresholds": {\n    "preserve_existing_probe_inheritance": true,\n    "source": "existing frozen arm-specific threshold contracts",\n    "test_refit_allowed": false\n  },\n  "timezone": "Asia/Bangkok",\n  "tokenizer": {\n    "chat_template": {\n      "add_generation_prompt": true,\n      "role": "user"\n    },\n    "gguf_tokenize": {\n      "add_bos": false,\n      "special": true\n    },\n    "repo": "Qwen/Qwen3-Coder-30B-A3B-Instruct",\n    "require_hf_gguf_token_id_equality": true,\n    "require_single_token_binary_labels": true,\n    "revision": "b2cff646eb4bb1d68355c01b18ae02e7cf42d120"\n  },\n  "user_decisions": {\n    "build": "approved: Kaggle platform build with mandatory complete provenance; no shared container required",\n    "cpu_threads": "approved: preserve default CPU-count formulas used by all 16 notebooks",\n    "gpu_policy": "layer + auto free VRAM, approved",\n    "inference_seed": 4294967295,\n    "packages": "exact locked versions approved, including scipy==1.16.3 on 2026-11-07"\n  }\n}\n'
RUNTIME_LOCK_PATH = Path('/kaggle/working/inference_runtime_lock.json')
if RUNTIME_LOCK_PATH.exists() and RUNTIME_LOCK_PATH.read_bytes() != RUNTIME_LOCK_BYTES:
    raise RuntimeError('Working directory has a different runtime contract; use a fresh session')
RUNTIME_LOCK_PATH.write_bytes(RUNTIME_LOCK_BYTES)
Path('/kaggle/working/grace_runtime_contract.py').write_text("import contextlib\nimport hashlib\nimport importlib.metadata\nimport json\nimport multiprocessing\nimport os\nimport platform\nimport re\nimport shutil\nimport subprocess\nimport sys\nfrom pathlib import Path\n\nEXPECTED_CONTRACT_SHA256 = 'bf1f859dc786e206a3c11d655b04b528dc68f59a15a82d4db894ac5944b0ac2e'\nCONTRACT_PATH = Path('/kaggle/working/inference_runtime_lock.json')\n\ndef digest(path):\n    h = hashlib.sha256()\n    with Path(path).open('rb') as f:\n        for block in iter(lambda: f.read(1024 * 1024), b''): h.update(block)\n    return h.hexdigest()\n\ndef load_contract(path=CONTRACT_PATH):\n    raw = Path(path).read_bytes()\n    if hashlib.sha256(raw).hexdigest() != EXPECTED_CONTRACT_SHA256:\n        raise ValueError('Runtime contract file SHA-256 mismatch')\n    c = json.loads(raw)\n    unsigned = {k:v for k,v in c.items() if k != 'integrity'}\n    seal = hashlib.sha256(json.dumps(unsigned,sort_keys=True,separators=(',',':'),ensure_ascii=False).encode()).hexdigest()\n    if seal != c['integrity']['sha256']:\n        raise ValueError('Runtime contract canonical integrity mismatch')\n    return c\n\ndef command(args):\n    p = subprocess.run(args, check=True, text=True, capture_output=True)\n    return p.stdout + p.stderr\n\ndef assert_runtime(c):\n    actual = {name:importlib.metadata.version(name) for name in c['packages']}\n    if actual != c['packages']: raise RuntimeError(f'Package mismatch: {actual}')\n    if platform.system() != 'Linux' or '.'.join(map(str,sys.version_info[:2])) != c['build']['python_family']:\n        raise RuntimeError('Contract requires Kaggle Linux/Python 3.13')\n    gpu = command(['nvidia-smi','--query-gpu=index,uuid,name,memory.free,memory.total,driver_version','--format=csv,noheader'])\n    lines = [s for s in gpu.splitlines() if s.strip()]\n    if len(lines) != c['gpu']['device_count'] or any('T4' not in s for s in lines):\n        raise RuntimeError(f'Contract requires 2 T4 GPUs: {gpu}')\n    dist = importlib.metadata.distribution('llama-cpp-python')\n    origin = json.loads(dist.read_text('direct_url.json') or '{}')\n    if origin.get('archive_info',{}).get('hashes',{}).get('sha256') != c['backend_source']['sdist_sha256']:\n        raise RuntimeError('Backend installation is not from the verified source archive')\n    return actual, gpu\n\ndef provenance(c, runner, target):\n    packages, gpu = assert_runtime(c)\n    dist = importlib.metadata.distribution('llama-cpp-python')\n    binaries = {}\n    for entry in dist.files or []:\n        if str(entry).endswith(('.so','.dll','.dylib')):\n            p = Path(dist.locate_file(entry)); binaries[str(entry)] = digest(p)\n    if not binaries: raise RuntimeError('No compiled backend library found for provenance')\n    result = dict(runtime_contract_sha256=EXPECTED_CONTRACT_SHA256,\n        runtime_contract_integrity=c['integrity']['sha256'], packages=packages,\n        python=sys.version, platform=platform.platform(), libc=platform.libc_ver(),\n        gpu_before_load=gpu, cpu_count=multiprocessing.cpu_count(),\n        n_threads=max(multiprocessing.cpu_count()//2,1), n_threads_batch=multiprocessing.cpu_count(),\n        constructor=c['constructor'], runner_sha256=digest(runner),\n        backend_sdist_sha256=c['backend_source']['sdist_sha256'], compiled_backend_sha256=binaries,\n        build_environment={k:os.environ.get(k) for k in ('CMAKE_ARGS','CMAKE_BUILD_PARALLEL_LEVEL','FORCE_CMAKE')})\n    for name,args in [('nvcc',['nvcc','--version']),('compiler',['cc','--version']),\n                      ('cxx_compiler',['c++','--version']),('cmake',['cmake','--version']),\n                      ('pip_freeze',[sys.executable,'-m','pip','freeze','--all'])]:\n        result[name] = command(args)\n    target = Path(target); target.parent.mkdir(parents=True,exist_ok=True)\n    target.write_text(json.dumps(result,indent=2),encoding='utf-8')\n    return result\n\ndef load_locked_model(c, model_path, tokenizer_manifest, gpu_layers, log_path):\n    import llama_cpp\n    if digest(model_path) != c['model']['sha256']: raise ValueError('Frozen model SHA-256 mismatch')\n    if gpu_layers != c['constructor']['n_gpu_layers']: raise ValueError('GPU layers differ from lock')\n    if (tokenizer_manifest['tokenizer'],tokenizer_manifest['tokenizer_revision'],tokenizer_manifest['prompt_budget_tokens']) != (\n        c['tokenizer']['repo'],c['tokenizer']['revision'],c['constructor']['n_ctx']):\n        raise ValueError('Tokenizer/context differ from runtime lock')\n    if not llama_cpp.llama_supports_gpu_offload(): raise RuntimeError('CUDA backend unavailable')\n    kwargs = dict(c['constructor'])\n    kwargs['split_mode'] = getattr(llama_cpp,kwargs['split_mode'])\n    kwargs.update(n_threads=max(multiprocessing.cpu_count()//2,1), n_threads_batch=multiprocessing.cpu_count())\n    # The pinned Python binding's logger writes through sys.stderr, so capture that stream directly.\n    path = Path(log_path)\n    with path.open('w',encoding='utf-8') as log, contextlib.redirect_stderr(log):\n        model = llama_cpp.Llama(model_path=str(model_path),verbose=True,**kwargs)\n    text = path.read_text(encoding='utf-8',errors='replace')\n    print(text,flush=True)\n    reports = re.findall(r'offloaded\\s+(\\d+)\\s*/\\s*(\\d+)\\s+layers?\\s+to\\s+GPU',text,re.I)\n    required = c['gpu']['required_offloaded_layers']\n    if not any(int(a)==int(b)==required for a,b in reports):\n        model.close()\n        raise RuntimeError(f'Full {required}/{required} GPU offload not verified; inspect {path}')\n    return model\n",encoding='utf-8')
if '/kaggle/working' not in sys.path: sys.path.insert(0,'/kaggle/working')
from grace_runtime_contract import load_contract, assert_runtime, EXPECTED_CONTRACT_SHA256
RUNTIME = load_contract()
shutil.copy2(RUNTIME_LOCK_PATH, DIAG / RUNTIME_LOCK_PATH.name)


In [ ]:
import math
ARM = 'B2-rand'
PROMPT_SEED = 44
KAGGLE_INPUT_ROOT = Path('/kaggle/input')
BUNDLE_WORK = Path('/kaggle/working/b2_rand_test_bundles')
OUT_ROOT = Path('/kaggle/working/b2_rand_seed44_test')
BUNDLE_NAME = 'grace_b2_rand_test_seed44_bundle.zip'
BUNDLE_SHA256 = 'e8bf81b3d5355a19221f7e61bbe19af81a31c953fd929c8e642918f67659590d'
CHECKPOINT_NAME = 'grace_b2_rand_seed44_test_checkpoint.zip'
CHECKPOINT_INPUT_PATH = None
MODEL_FILENAME = 'Qwen3-Coder-30B-A3B-Instruct-Q4_K_M.gguf'
MODEL_SHA256 = 'f6050b4da22869aa3fb9d94bb3314e9fc5a027d4741c1f97761c387d6b2b2335'
MAX_RUNTIME_HOURS = 9
CHECKPOINT_RESERVE_SECONDS = 300
GPU_LAYERS = -1
THRESHOLD_LOCK_SHA256 = '48c12066bbddabc8ef674a70f6f9ac68cb74cbeecc2a5ebbbda3cb4cb6498735'
SETS = [
    {'name':'seed_44/devign_test.zip','sha256':'82af62d88f797cffc9531efa26ca501584f672845826864d1f8ca9cbab708eb7','seed':44,'dataset':'devign','count':2732},
    {'name':'seed_44/reveal_test.zip','sha256':'931ed9895690d888b4a23913a61296b58764a0657b375e3806a0eb29f09f40db','seed':44,'dataset':'reveal','count':2274},
]
EXPECTED_OUTER_MEMBER_HASHES = {'delivery_manifest.json': '2012fb7d84df0514625283f68566f9d4ea43a5ac24824aaf6fed783bb0d91971', 'seed_44/devign_test.zip': '82af62d88f797cffc9531efa26ca501584f672845826864d1f8ca9cbab708eb7', 'seed_44/reveal_test.zip': '931ed9895690d888b4a23913a61296b58764a0657b375e3806a0eb29f09f40db', 'threshold_lock.json': 'f5bb21ad64853950b1eb3ba534aaca3b35e025c20520e3c9af89a0d9b5499d20'}
EXPECTED_INNER_MEMBER_HASHES = {'seed_44/devign_test.zip': {'bundle_manifest.json': '177744f9c3b2ebe0f51a53822c4667c597a22eaa3e2d98ea23e4adf660093f7d', 'prompts.jsonl': 'b3ef2ef95ccb5d99b739ed6fbf30aa876f170f8d2def26d48ebe3eece81261d9', 'threshold_lock.json': 'f5bb21ad64853950b1eb3ba534aaca3b35e025c20520e3c9af89a0d9b5499d20'}, 'seed_44/reveal_test.zip': {'bundle_manifest.json': '3f70390f6093ef739c73d3b53e5f7af9913c3d81119e39d79bd75dcf94f74e10', 'prompts.jsonl': '8e35d22c5d1413d67200730fec4ce58270d2d7250a84f572394d024ffe825785', 'threshold_lock.json': 'f5bb21ad64853950b1eb3ba534aaca3b35e025c20520e3c9af89a0d9b5499d20'}}


In [ ]:
def hash_bytes(data):
    return hashlib.sha256(data).hexdigest()

BUNDLE_WORK.mkdir(parents=True, exist_ok=True)
OUT_ROOT.mkdir(parents=True, exist_ok=True)
archives = sorted(p for p in KAGGLE_INPUT_ROOT.rglob(BUNDLE_NAME) if p.is_file())
if len(archives) > 1: raise ValueError(f'Multiple delivery bundles found: {archives}')
payloads = {}
if archives:
    if hash_bytes(archives[0].read_bytes()) != BUNDLE_SHA256: raise ValueError('Outer bundle SHA-256 mismatch')
    with zipfile.ZipFile(archives[0]) as outer:
        if set(outer.namelist()) != set(EXPECTED_OUTER_MEMBER_HASHES): raise ValueError('Unexpected bundle member list')
        payloads = {name:outer.read(name) for name in EXPECTED_OUTER_MEMBER_HASHES}
else:
    # Kaggle sometimes unpacks both the outer bundle and each nested test ZIP.
    for name,digest in EXPECTED_OUTER_MEMBER_HASHES.items():
        candidates=sorted(p for p in KAGGLE_INPUT_ROOT.rglob(Path(name).name) if p.is_file())
        if name.endswith('.zip'):
            valid=[p for p in candidates if hash_bytes(p.read_bytes())==digest]
            if valid: payloads[name]=valid[0].read_bytes(); continue
            extracted=[]
            parts=Path(name).parts; folder_name=Path(parts[-1]).stem
            for manifest_path in KAGGLE_INPUT_ROOT.rglob('bundle_manifest.json'):
                folder=manifest_path.parent
                if folder.name != folder_name or folder.parent.name != parts[0]: continue
                members={m:folder/m for m in EXPECTED_INNER_MEMBER_HASHES[name]}
                if all(p.is_file() and hash_bytes(p.read_bytes())==EXPECTED_INNER_MEMBER_HASHES[name][m] for m,p in members.items()):
                    extracted.append({m:p.read_bytes() for m,p in members.items()})
            if extracted: payloads[name]=extracted[0]
            else: raise FileNotFoundError(f'Missing or hash-mismatched nested test bundle {name}; files={[str(x) for x in candidates]}')
        else:
            valid=[p for p in candidates if hash_bytes(p.read_bytes())==digest]
            if not valid: raise FileNotFoundError(f'Missing or hash-mismatched {name}')
            payloads[name]=valid[0].read_bytes()

for name,digest in EXPECTED_OUTER_MEMBER_HASHES.items():
    payload=payloads[name]
    if isinstance(payload,bytes) and hash_bytes(payload)!=digest: raise ValueError(f'Outer member hash mismatch: {name}')

bundles={}; expected={}; locked_thresholds={}
lock_payload=json.loads(payloads['threshold_lock.json'])
seal=lock_payload.get('locked_sha256')
actual=hash_bytes(json.dumps({k:v for k,v in lock_payload.items() if k!='locked_sha256'},sort_keys=True,separators=(',',':'),ensure_ascii=False).encode())
if seal!=actual or seal!=THRESHOLD_LOCK_SHA256: raise ValueError('Threshold lock signature mismatch')
for spec in SETS:
    name=spec['name']; raw=payloads[name]; ds=spec['dataset']; seed=spec['seed']
    path=BUNDLE_WORK/Path(name).name
    if isinstance(raw,dict):
        inner_members=raw
        with zipfile.ZipFile(path,'w',compression=zipfile.ZIP_DEFLATED) as normalized:
            for member,data in inner_members.items(): normalized.writestr(member,data)
    else:
        if hash_bytes(raw)!=spec['sha256']: raise ValueError(f'Inner ZIP hash mismatch: {name}')
        path.write_bytes(raw)
        with zipfile.ZipFile(path) as inner: inner_members={m:inner.read(m) for m in EXPECTED_INNER_MEMBER_HASHES[name]}
    for member,digest in EXPECTED_INNER_MEMBER_HASHES[name].items():
        if hash_bytes(inner_members[member])!=digest: raise ValueError(f'Extracted member hash mismatch: {name}/{member}')
    prompt_bytes=inner_members['prompts.jsonl']; manifest=json.loads(inner_members['bundle_manifest.json'])
    embedded_lock=json.loads(inner_members['threshold_lock.json'])
    if embedded_lock.get('locked_sha256')!=seal or manifest.get('threshold_lock_sha256')!=seal: raise ValueError('Inner lock/manifest mismatch')
    key=f'B2-rand/{seed}/{ds}'; entry=lock_payload['entries'][key]; locked_thresholds[seed,ds]=entry['tau_star']
    if (manifest.get('arm'),manifest.get('seed'),manifest.get('dataset'),manifest.get('split'))!=('B2-rand',44,ds,'test'): raise ValueError('Wrong test bundle identity')
    if manifest.get('tau_star')!=entry['tau_star'] or manifest.get('model_sha256')!=MODEL_SHA256: raise ValueError('Locked validation threshold/model mismatch')
    if hash_bytes(prompt_bytes)!=manifest['prompts_sha256']: raise ValueError('Prompt JSONL hash mismatch')
    rows=[json.loads(line) for line in prompt_bytes.decode().splitlines() if line.strip()]
    if len(rows)!=spec['count'] or len({str(r['id']) for r in rows})!=spec['count']: raise ValueError('Wrong row count or duplicate IDs')
    for row in rows:
        if (row.get('arm'),row.get('seed'),row.get('dataset'),row.get('split'))!=('B2-rand',44,ds,'test'): raise ValueError('Wrong prompt identity')
        if hash_bytes(row['prompt'].encode())!=row['prompt_sha256']: raise ValueError(f'Prompt hash mismatch: {row["id"]}')
    bundles[seed,ds]=path; expected[seed,ds]={str(r['id']):(r['prompt_sha256'],r['label']) for r in rows}


# Load only the checkpoint from this seed, if one is attached. Its entries are
# copied under the same OUT_ROOT layout and then checked by validate_records().
allowed = {f'seed_{PROMPT_SEED}/{ds}/test/{name}' for ds in ('devign','reveal') for name in ('records.jsonl','run_manifest.json')}
if CHECKPOINT_INPUT_PATH is not None:
    checkpoint = Path(CHECKPOINT_INPUT_PATH)
else:
    matches = sorted(KAGGLE_INPUT_ROOT.rglob(CHECKPOINT_NAME))
    if len(matches) > 1: raise ValueError('Multiple seed checkpoints attached; specify CHECKPOINT_INPUT_PATH')
    checkpoint = matches[0] if matches else None
if checkpoint:
    with zipfile.ZipFile(checkpoint) as source:
        names = [n for n in source.namelist() if n in allowed]
        if not any(n.endswith('records.jsonl') for n in names): raise ValueError('Checkpoint has no records for this seed')
        for name in names:
            target = OUT_ROOT / name
            target.parent.mkdir(parents=True,exist_ok=True)
            target.write_bytes(source.read(name))

def validate_records():
    progress = {}
    for (seed,ds), prompts in expected.items():
        folder = OUT_ROOT / f'seed_{seed}' / ds / 'test'
        path = folder / 'records.jsonl'
        seen = set()
        if path.exists():
            if not (folder/'run_manifest.json').is_file(): raise ValueError(f'Missing manifest: {path}')
            for line in path.read_text(encoding='utf-8').splitlines():
                if not line.strip(): continue
                row = json.loads(line); uid = str(row['id'])
                lock_entry = lock_payload['entries'][f'B2-rand/{seed}/{ds}']
                if (uid in seen or (row.get('arm'),row.get('seed'),row.get('dataset'),row.get('split'),row.get('model_sha256')) != ('B2-rand',seed,ds,'test',MODEL_SHA256)
                    or prompts.get(uid) != (row.get('prompt_sha256'),row.get('label'))
                    or not math.isfinite(row.get('s',float('nan')))
                    or row.get('threshold_lock_sha256') != THRESHOLD_LOCK_SHA256
                    or row.get('tau') != lock_entry['tau_star']
                    or row.get('pred_tau') != int(row['s'] >= row['tau'])
                    or row.get('inference_seed') != RUNTIME['constructor']['seed']
                    or row.get('runtime_contract_sha256') != EXPECTED_CONTRACT_SHA256):
                    raise ValueError(f'Unexpected checkpoint row: {uid}')
                seen.add(uid)
        progress[seed,ds] = len(seen)
    return progress

if (OUT_ROOT/'threshold_lock.json').exists():
    if hash_bytes((OUT_ROOT/'threshold_lock.json').read_bytes()) != hash_bytes(json.dumps(lock_payload,indent=2,ensure_ascii=False).encode()):
        raise ValueError('Existing threshold lock differs from the attached test bundle')
else:
    (OUT_ROOT/'threshold_lock.json').write_text(json.dumps(lock_payload,indent=2,ensure_ascii=False),encoding='utf-8')

shutil.copy2(RUNTIME_LOCK_PATH, OUT_ROOT / RUNTIME_LOCK_PATH.name)


In [ ]:
# Install packages from the runtime lock and verify the backend source.
import urllib.request

build = RUNTIME["build"]
os.environ["CMAKE_ARGS"] = (
    "-DGGML_CUDA=on "
    f"-DCMAKE_CUDA_ARCHITECTURES={build['CMAKE_CUDA_ARCHITECTURES']} "
    f"-DCMAKE_LIBRARY_PATH={build['CMAKE_LIBRARY_PATH']}"
)
for key in ("CMAKE_BUILD_PARALLEL_LEVEL", "FORCE_CMAKE"):
    os.environ[key] = build[key]

packages = [
    f"{name}=={version}"
    for name, version in RUNTIME["packages"].items()
    if name != "llama-cpp-python"
]
packages += ["typing-extensions>=4.5.0", "diskcache>=5.6.1", "jinja2>=2.11.3"]
run_logged(
    [sys.executable, "-m", "pip", "install", "--no-cache-dir", *packages],
    "install_dependencies",
)

source = RUNTIME["backend_source"]
url = f"https://pypi.org/pypi/llama-cpp-python/{source['version']}/json"
with urllib.request.urlopen(url) as response:
    metadata = json.load(response)

artifact = next(
    item for item in metadata["urls"]
    if item["filename"] == source["sdist_filename"]
)
if artifact["digests"]["sha256"] != source["sdist_sha256"]:
    raise ValueError("Backend source registry hash mismatch")

archive_path = Path("/kaggle/working") / source["sdist_filename"]
urllib.request.urlretrieve(artifact["url"], archive_path)
if hashlib.sha256(archive_path.read_bytes()).hexdigest() != source["sdist_sha256"]:
    raise ValueError("Backend source bytes hash mismatch")

run_logged(
    [
        sys.executable, "-m", "pip", "install", "-v", "--no-cache-dir",
        "--no-deps", "--force-reinstall", str(archive_path),
    ],
    "install_llama_cuda",
)
assert_runtime(RUNTIME)
versions()
diagnostics_zip()


In [ ]:
assert_runtime(RUNTIME)
import llama_cpp
versions()
note('llama-cpp-python GPU offload: ' + str(llama_cpp.llama_supports_gpu_offload()))
assert llama_cpp.llama_supports_gpu_offload(), 'llama-cpp-python chưa có CUDA; xem install_llama_cuda.log'
diagnostics_zip()


In [ ]:
# Tìm đúng tên file model trong toàn bộ Input đã gắn, bất kể username/slug.
model_files = sorted(path for path in KAGGLE_INPUT_ROOT.rglob(MODEL_FILENAME) if path.is_file())
if len(model_files) != 1:
    raise FileNotFoundError(f'Cần đúng một {MODEL_FILENAME} trong Kaggle Input; '
                            f'tìm thấy: {list(map(str, model_files))}. '
                            'Hãy Add Input chứa model và bỏ bản trùng nếu có; '
                            'notebook không tự tải lại GGUF từ Hugging Face.')
model_path = str(model_files[0])
revision = 'local-input'
note(f'Dùng model input: {model_path}')
print('Model input:', model_path)
print('Batch output directory:', OUT_ROOT)


In [ ]:
%%writefile /kaggle/working/kaggle_inference.py
import argparse
import hashlib
import json
import math
import os
import platform
import time
import zipfile
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
from grace_runtime_contract import load_contract, provenance, load_locked_model, EXPECTED_CONTRACT_SHA256
RUNTIME = load_contract()

EXPECTED_LOCK_SEAL = '48c12066bbddabc8ef674a70f6f9ac68cb74cbeecc2a5ebbbda3cb4cb6498735'


def sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()


def write_json_atomic(path, payload):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + '.tmp')
    tmp.write_text(json.dumps(payload, indent=2), encoding='utf-8')
    os.replace(tmp, path)


def binary_scores(logits, label_ids):
    values = np.asarray(logits, dtype=np.float64).reshape(-1)
    if not np.all(np.isfinite(values)):
        raise ValueError('Non-finite logits')
    maximum = np.max(values)
    normalizer = float(maximum + np.log(np.exp(values - maximum).sum()))
    lp0 = float(values[label_ids[0]] - normalizer)
    lp1 = float(values[label_ids[1]] - normalizer)
    score = float(np.exp(lp1 - np.logaddexp(lp0, lp1)))
    best = int(np.argmax(values))
    top = np.argsort(values)[-5:][::-1]
    return {'logprob_0': lp0, 'logprob_1': lp1, 'p0': math.exp(lp0), 'p1': math.exp(lp1),
            's': score, 'pred_argmax': int(score >= 0.5), 'greedy_token_id': best,
            'top5': [{'token_id': int(i), 'logprob': float(values[i] - normalizer)} for i in top]}


def load_bundle(bundle):
    with zipfile.ZipFile(bundle) as z:
        manifest = json.loads(z.read('bundle_manifest.json'))
        prompt_bytes = z.read('prompts.jsonl')
        lock = json.loads(z.read('threshold_lock.json'))
    seal = lock.get('locked_sha256')
    unsigned = {k:v for k,v in lock.items() if k!='locked_sha256'}
    actual = hashlib.sha256(json.dumps(unsigned,sort_keys=True,separators=(',',':'),ensure_ascii=False).encode()).hexdigest()
    if seal != actual or seal != EXPECTED_LOCK_SEAL or manifest.get('threshold_lock_sha256') != seal:
        raise ValueError('B2-rand validation lock integrity mismatch')
    ds, seed = manifest['dataset'], manifest['seed']
    entry = lock['entries'][f"B2-rand/{seed}/{ds}"]
    if (manifest.get('arm'), manifest.get('split')) != ('B2-rand','test'):
        raise ValueError('Only B2-rand test bundles are accepted')
    if manifest.get('tau_star') != entry['tau_star'] or lock.get('model_sha256') != manifest.get('model_sha256'):
        raise ValueError('Test bundle does not use its locked validation threshold/model')
    if hashlib.sha256(prompt_bytes).hexdigest() != manifest['prompts_sha256']:
        raise ValueError(f'Prompt bundle hash mismatch: {bundle}')
    rows = [json.loads(line) for line in prompt_bytes.decode('utf-8').splitlines() if line.strip()]
    if len(rows) != manifest['count'] or len({str(r['id']) for r in rows}) != len(rows):
        raise ValueError('Prompt count/unique ID mismatch')
    for row in rows:
        if (row.get('arm'),row.get('seed'),row.get('dataset'),row.get('split')) != ('B2-rand',seed,ds,'test'):
            raise ValueError('Wrong test prompt identity')
        if hashlib.sha256(row['prompt'].encode('utf-8')).hexdigest() != row['prompt_sha256']:
            raise ValueError('Test prompt hash mismatch')
    manifest['model_repo'] = 'ggml-org/Qwen3-Coder-30B-A3B-Instruct-GGUF'
    manifest['prompt_budget_tokens'] = 8192
    return manifest, rows


def evaluate_output(output):
    from sklearn.metrics import (accuracy_score, average_precision_score, balanced_accuracy_score,
                                 confusion_matrix, f1_score, matthews_corrcoef, precision_score,
                                 recall_score, roc_auc_score)
    records = [json.loads(line) for line in output.read_text(encoding='utf-8').splitlines() if line.strip()]
    rows = [r for r in records if r['valid'] and math.isfinite(r['s'])]
    if not rows: raise ValueError('No valid test scores')
    y = [r['label'] for r in rows]
    pred = [r['pred_tau'] for r in rows]
    scores = [r['s'] for r in rows]
    # Score-valid rows remain included even when the raw greedy token is not 0/1.
    return dict(n_total=len(records), n_score_valid=len(rows),
                n_response_valid=sum(r['response_valid'] for r in records),
                tau=rows[0]['tau'], threshold_lock_sha256=EXPECTED_LOCK_SEAL,
                accuracy=accuracy_score(y,pred), balanced_accuracy=balanced_accuracy_score(y,pred),
                precision=precision_score(y,pred,zero_division=0), recall=recall_score(y,pred,zero_division=0),
                f1=f1_score(y,pred,zero_division=0),
                mcc=matthews_corrcoef(y,pred) if len(set(y))==2 and len(set(pred))==2 else None,
                confusion_matrix=confusion_matrix(y,pred,labels=[0,1]).tolist(),
                auroc=roc_auc_score(y,scores) if len(set(y))==2 else None,
                average_precision=average_precision_score(y,scores) if len(set(y))==2 else None,
                threshold_policy='inherit_completed_Kaggle_validation_no_test_fit')


def snapshot(status_path):
    root = Path(status_path).parent
    target = root.parent / ('grace_' + root.name + '_checkpoint.zip')
    temporary = target.with_suffix('.zip.tmp')
    with zipfile.ZipFile(temporary, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(root.rglob('*')):
            if path.is_file() and not path.name.endswith('.tmp'):
                archive.write(path, path.relative_to(root).as_posix())
    os.replace(temporary, target)


def run_batch(runs, model_path, model_revision, gpu_layers, limit, deadline_epoch,
              checkpoint_reserve_seconds, status_path, prompt_seed, expected_model_sha256, arm):
    import llama_cpp
    from llama_cpp import Llama, __version__ as llama_version, LLAMA_DEFAULT_SEED
    from transformers import AutoTokenizer
    if llama_version != '0.3.16': raise ValueError(f'Backend version mismatch: {llama_version}')
    inference_seed = RUNTIME['constructor']['seed']

    if arm != 'B2-rand' or prompt_seed != 44 or limit is not None or not (1 <= len(runs) <= 2) or any(r[1] != 'test' for r in runs):
        raise ValueError('Only complete B2-rand test census for prompt seed 44 is allowed')
    started = time.time()
    durations = []
    last_snapshot = started
    stop_at = deadline_epoch - checkpoint_reserve_seconds
    status_path = Path(status_path)
    status_path.parent.mkdir(parents=True, exist_ok=True)
    status = {'state': 'starting', 'timed_out': False,
              'started_at': datetime.fromtimestamp(started, timezone.utc).isoformat(),
              'deadline_at': datetime.fromtimestamp(deadline_epoch, timezone.utc).isoformat(),
              'stop_new_records_at': datetime.fromtimestamp(stop_at, timezone.utc).isoformat(),
              'max_runtime_hours': (deadline_epoch - started) / 3600,
              'checkpoint_reserve_seconds': checkpoint_reserve_seconds,
              'runs': {}, 'completed_records_total': 0}
    for dataset, split, bundle, out in runs:
        key = f'{dataset}/{split}'
        status['runs'][key] = {'state': 'pending', 'bundle': bundle, 'output': out}
    write_json_atomic(status_path, status)

    def stop_for_deadline(current=None):
        status['state'] = 'timed_out'
        status['timed_out'] = True
        status['timed_out_at'] = datetime.now(timezone.utc).isoformat()
        status['stopped_after_run'] = current
        write_json_atomic(status_path, status)

    if time.time() >= stop_at:
        stop_for_deadline(None)
        return

    model_path = Path(model_path)
    model_digest = sha256(model_path)
    if model_digest != expected_model_sha256: raise ValueError('Model SHA256 không khớp contract')
    if model_revision == 'local-input':
        model_revision = f'local-sha256:{model_digest}'
    first_manifest, _ = load_bundle(runs[0][2])
    for dataset, split, bundle, out in runs:
        current_manifest, _ = load_bundle(bundle)
        if (current_manifest['tokenizer'], current_manifest['tokenizer_revision'],
            current_manifest['prompt_budget_tokens']) != (first_manifest['tokenizer'],
            first_manifest['tokenizer_revision'], first_manifest['prompt_budget_tokens']):
            raise ValueError('All four bundles must use the same tokenizer revision and context budget')
    tokenizer = AutoTokenizer.from_pretrained(first_manifest['tokenizer'],
                                                revision=first_manifest['tokenizer_revision'])
    runtime_info = provenance(RUNTIME, __file__, status_path.parent / 'runtime_provenance.json')
    llm = load_locked_model(RUNTIME, model_path, first_manifest, gpu_layers, status_path.parent / 'model_load_offload.log')
    runtime_info['gpu_after_load'] = __import__('grace_runtime_contract').command(['nvidia-smi','--query-gpu=index,uuid,name,memory.free,memory.total,driver_version','--format=csv,noheader'])
    runtime_info['offload_log_sha256'] = sha256(status_path.parent / 'model_load_offload.log')
    (status_path.parent / 'runtime_provenance.json').write_text(json.dumps(runtime_info,indent=2),encoding='utf-8')
    if time.time() >= stop_at:
        stop_for_deadline(None)
        return
    label_ids = []
    for label in ('0', '1'):
        hf_ids = tokenizer.encode(label, add_special_tokens=False)
        gguf_ids = llm.tokenize(label.encode(), add_bos=False, special=False)
        if len(hf_ids) != 1 or hf_ids != gguf_ids:
            raise ValueError(f'Label token mismatch for {label}: HF={hf_ids}, GGUF={gguf_ids}')
        label_ids.append(hf_ids[0])
    if label_ids[0] == label_ids[1]:
        raise ValueError('Label token IDs must differ')

    status['state'] = 'running'
    write_json_atomic(status_path, status)
    for dataset, split, bundle, out_text in runs:
        key = f'{dataset}/{split}'
        out = Path(out_text)
        out.mkdir(parents=True, exist_ok=True)
        manifest, rows = load_bundle(bundle)
        if (manifest.get('arm'), manifest.get('seed'), manifest['dataset'], manifest['split']) != (arm, prompt_seed, dataset, split):
            raise ValueError(f'Bundle identity mismatch for {key}: {manifest["dataset"]}/{manifest["split"]}')
        if limit is not None:
            rows = rows[:limit]
        identity = {'bundle_sha256': sha256(bundle),
                    'prompt_set_sha256': manifest['prompts_sha256'], 'model_sha256': model_digest,
                    'model_repo': manifest['model_repo'], 'quantization': manifest['quantization'],
                    'model_file': model_path.name, 'llama_cpp_python': llama_version,
                    'model_repo_revision': model_revision, 'tokenizer': manifest['tokenizer'],
                    'tokenizer_revision': manifest['tokenizer_revision'], 'gpu_layers': gpu_layers,
                    'context_tokens': manifest['prompt_budget_tokens'],
                    'scoring': 'unconstrained_first_token_logits_v1', 'limit': limit,
                    'python': platform.python_version(), 'dataset': dataset, 'split': split, 'arm': arm, 'prompt_seed': prompt_seed,
                    'inference_seed': inference_seed, 'inference_seed_policy': 'llama_cpp_default',
                    'threshold_lock_sha256': EXPECTED_LOCK_SEAL, 'tau_star': manifest['tau_star'],
                    'n_batch': RUNTIME['constructor']['n_batch'], 'runtime_contract_sha256':EXPECTED_CONTRACT_SHA256, 'runtime_constructor':RUNTIME['constructor'], 'runtime_packages':RUNTIME['packages'], 'backend_sdist_sha256':RUNTIME['backend_source']['sdist_sha256']}
        identity_file = out / 'run_manifest.json'
        previous_identity = (json.loads(identity_file.read_text(encoding='utf-8'))
                             if identity_file.exists() else None)
        if previous_identity is not None:
            # Ignore only archive-byte hashes: ZIP timestamps/compression metadata may change.
            # Prompt identity is verified below against every checkpoint row's prompt_sha256.
            ignored = {'bundle_sha256', 'prompt_set_sha256'}
            previous_stable = {k: v for k, v in previous_identity.items() if k not in ignored}
            current_stable = {k: v for k, v in identity.items() if k not in ignored}
            if previous_stable != current_stable:
                raise ValueError(f'Existing output has different model/run settings for {key}; use a new OUT_ROOT')
            if ('prompt_set_sha256' in previous_identity and
                    previous_identity['prompt_set_sha256'] != identity['prompt_set_sha256']):
                raise ValueError(f'Prompt bundle content changed for {key}; refusing to resume')
        output = out / 'records.jsonl'
        done = {}
        if output.exists():
            for line in output.read_text(encoding='utf-8').splitlines():
                if line.strip():
                    row = json.loads(line)
                    row_id = str(row['id'])
                    if row_id in done:
                        raise ValueError(f'Duplicate checkpoint ID in {key}: {row_id}')
                    done[row_id] = row
        expected_ids = {str(item['id']) for item in rows}
        unexpected_ids = set(done) - expected_ids
        if unexpected_ids:
            raise ValueError(f'Checkpoint contains IDs absent from current bundle {key}: {sorted(unexpected_ids)[:10]}')
        for item in rows:
            row_id = str(item['id'])
            if row_id in done:
                if done[row_id].get('prompt_sha256') != item['prompt_sha256']:
                    raise ValueError(f'Checkpoint prompt mismatch: {key}/{row_id}')
                saved = done[row_id]
                if (saved.get('threshold_lock_sha256') != EXPECTED_LOCK_SEAL or saved.get('tau') != manifest['tau_star']
                        or saved.get('label') != item['label'] or not saved.get('valid')
                        or not all(math.isfinite(saved[k]) for k in ('s','logprob_0','logprob_1'))
                        or saved.get('pred_tau') != int(saved['s'] >= manifest['tau_star'])):
                    raise ValueError('Invalid test checkpoint identity or score: ' + row_id)
                if done[row_id].get('model_sha256') != model_digest:
                    raise ValueError(f'Checkpoint model hash mismatch: {key}/{row_id}')
        # Persist the refreshed archive hash only after checkpoint IDs and prompt hashes pass.
        identity_file.write_text(json.dumps(identity, indent=2), encoding='utf-8')
        progress = status['runs'][key]
        progress.update(state='running', count=len(rows), completed=len(done),
                        records_path=str(output), bundle_sha256=identity['bundle_sha256'])
        status['completed_records_total'] = sum(x.get('completed', 0) for x in status['runs'].values())
        write_json_atomic(status_path, status)
        if len(done) > len(rows):
            raise ValueError(f'Checkpoint has more rows than this bundle: {key}')
        print(f'{key}: verified checkpoint IDs={len(done)}; skipping these; remaining={len(rows) - len(done)}', flush=True)

        with output.open('a', encoding='utf-8', newline='\n') as f:
            for item in rows:
                row_id = str(item['id'])
                if row_id in done:
                    if done[row_id]['prompt_sha256'] != item['prompt_sha256']:
                        raise ValueError(f'Checkpoint prompt mismatch: {key}/{row_id}')
                    continue
                if time.time() + max(30, 2 * max(durations[-20:], default=30)) >= stop_at:
                    progress.update(state='checkpointed', completed=len(done))
                    status['completed_records_total'] = sum(x.get('completed', 0) for x in status['runs'].values())
                    stop_for_deadline(key)
                    return
                prompt = item['prompt']
                if hashlib.sha256(prompt.encode()).hexdigest() != item['prompt_sha256']:
                    raise ValueError(f'Prompt hash mismatch: {key}/{row_id}')
                messages = [{'role': 'user', 'content': prompt}]
                chat = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
                hf_ids = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_dict=False)
                gguf_ids = llm.tokenize(chat.encode('utf-8'), add_bos=False, special=True)
                if list(hf_ids) != gguf_ids:
                    raise ValueError(f'Tokenizer mismatch at {key}/{row_id}; abort rather than alter prompt')
                if len(gguf_ids) > manifest['prompt_budget_tokens']:
                    raise ValueError(f'Token budget exceeded at {key}/{row_id}')
                start = time.monotonic()
                llm.reset()
                llm.eval(gguf_ids)
                scores = binary_scores(llm.scores[llm.n_tokens - 1], label_ids)
                raw = llm.detokenize([scores['greedy_token_id']]).decode('utf-8', errors='replace').strip()
                row = {k: item[k] for k in ('id', 'dataset', 'split', 'label', 'prompt_sha256')}
                row.update({k: item[k] for k in ('demo_ids', 'tokens', 'graph_missing', 'prompt_variant') if k in item})
                row.update(scores, arm=arm, raw=raw, response_valid=raw in ('0', '1'), valid=True,
                           token_ids={'0': label_ids[0], '1': label_ids[1]}, model_sha256=model_digest,
                           backend='llama-cpp-python', quantization='Q4_K_M', model_repo=manifest['model_repo'],
                           model_repo_revision=model_revision, seed=prompt_seed, inference_seed=inference_seed, timestamp=datetime.now(timezone.utc).isoformat(),
                           logprob_distribution='unconstrained_first_token', elapsed_s=time.monotonic()-start)
                row.update(runtime_contract_sha256=EXPECTED_CONTRACT_SHA256)
                row.update(tau=manifest['tau_star'], pred_tau=int(scores['s'] >= manifest['tau_star']),
                           threshold_lock_sha256=EXPECTED_LOCK_SEAL)
                durations.append(row['elapsed_s'])
                f.write(json.dumps(row, ensure_ascii=False, allow_nan=False) + '\n')
                f.flush()
                os.fsync(f.fileno())
                done[row_id] = row
                progress['completed'] = len(done)
                status['completed_records_total'] = sum(x.get('completed', 0) for x in status['runs'].values())
                status['last_checkpoint_at'] = datetime.now(timezone.utc).isoformat()
                write_json_atomic(status_path, status)
                if len(done) % 100 == 0 or time.time() - last_snapshot >= 120:
                    snapshot(status_path)
                    last_snapshot = time.time()
                print(f'{key} {row_id}: s={scores["s"]:.4f}, raw={raw!r} ({len(done)}/{len(rows)})', flush=True)
                if time.time() >= stop_at:
                    progress['state'] = 'checkpointed'
                    stop_for_deadline(key)
                    return

        metrics = evaluate_output(output)
        (out / 'evaluation.json').write_text(json.dumps(metrics, indent=2, allow_nan=False), encoding='utf-8')
        progress.update(state='complete', completed=len(rows), evaluation=str(out / 'evaluation.json'))
        status['completed_records_total'] = sum(x.get('completed', 0) for x in status['runs'].values())
        write_json_atomic(status_path, status)

    status['state'] = 'complete'
    status['timed_out'] = False
    status['completed_at'] = datetime.now(timezone.utc).isoformat()
    write_json_atomic(status_path, status)


if __name__ == '__main__':
    ap = argparse.ArgumentParser()
    ap.add_argument('--model', required=True)
    ap.add_argument('--prompt-seed', type=int, required=True, help='Frozen prompt-set seed; inference uses llama.cpp default seed')
    ap.add_argument('--arm', required=True)
    ap.add_argument('--expected-model-sha256', required=True)
    ap.add_argument('--model-revision', required=True)
    ap.add_argument('--gpu-layers', type=int, default=-1)
    ap.set_defaults(limit=None)
    ap.add_argument('--deadline-epoch', type=float, required=True)
    ap.add_argument('--checkpoint-reserve-seconds', type=int, default=300)
    ap.add_argument('--status', required=True)
    ap.add_argument('--run', action='append', nargs=4, metavar=('DATASET', 'SPLIT', 'BUNDLE', 'OUT'), required=True)
    args = ap.parse_args()
    run_batch(args.run, args.model, args.model_revision, args.gpu_layers, args.limit,
              args.deadline_epoch, args.checkpoint_reserve_seconds, args.status, args.prompt_seed, args.expected_model_sha256, args.arm)


In [ ]:
deadline = SESSION_DEADLINE_EPOCH
state = 'error'
progress = validate_records()
try:
    for seed in (PROMPT_SEED,):
        needed = [ds for ds in ('devign', 'reveal')
                  if (seed, ds) in bundles and progress[seed, ds] < len(expected[seed, ds])]
        if not needed:
            continue
        if time.time() >= deadline - CHECKPOINT_RESERVE_SECONDS:
            break
        status = OUT_ROOT / f'seed_{seed}_status.json'
        command = [sys.executable, '-u', '/kaggle/working/kaggle_inference.py',
                   '--model', model_path, '--model-revision', revision,
                   '--prompt-seed', str(seed), '--arm', ARM, '--expected-model-sha256', MODEL_SHA256,
                   '--gpu-layers', str(GPU_LAYERS), '--deadline-epoch', str(deadline),
                   '--checkpoint-reserve-seconds', str(CHECKPOINT_RESERVE_SECONDS), '--status', str(status)]
        for ds in needed:
            command += ['--run', ds, 'test', str(bundles[seed, ds]),
                        str(OUT_ROOT / f'seed_{seed}' / ds / 'test')]
        run_logged(command, f'test_seed_{seed}')
        progress = validate_records()
        result = json.loads(status.read_text(encoding='utf-8'))
        if result['timed_out']:
            break
        if any(progress[seed, ds] != len(expected[seed, ds]) for ds in needed):
            raise RuntimeError('Test stopped without a timeout or complete output')
    complete = all(progress[key] == len(expected[key]) for key in expected)
    import runpy
    evaluation_helper = runpy.run_path('/kaggle/working/kaggle_inference.py')['evaluate_output']
    for (seed, ds), count in progress.items():
        if count == len(expected[seed, ds]):
            folder = OUT_ROOT / f'seed_{seed}' / ds / 'test'
            metrics = evaluation_helper(folder / 'records.jsonl')
            (folder/'evaluation.json').write_text(json.dumps(metrics, indent=2, allow_nan=False), encoding='utf-8')
            print(ds, 'fixed-validation-threshold metrics:', metrics)
    state = 'test_complete' if complete else 'timed_out'
finally:
    progress = validate_records()
    summary = {'runtime_contract_sha256':EXPECTED_CONTRACT_SHA256, 'threshold_lock_sha256': THRESHOLD_LOCK_SHA256, 'thresholds': {ds: locked_thresholds.get((PROMPT_SEED, ds)) for ds in ('devign','reveal')}, 'arm': ARM, 'state': state, 'split': 'test', 'bundle_sha256': BUNDLE_SHA256,
               'progress': {f'{seed}/{ds}': progress[seed, ds] for seed, ds in progress}}
    (OUT_ROOT / 'batch_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
    archive = shutil.make_archive(str(Path('/kaggle/working') / CHECKPOINT_NAME.removesuffix('.zip')),
                                  'zip', root_dir=OUT_ROOT)
    print(state, archive)
    diagnostics_zip()
